In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


In [0]:
torch.manual_seed(42)

In [0]:
df = pd.read_csv(r"data/fmnist_small.csv")
df.head()

In [0]:
fig, axes = plt.subplots(4, 4, figsize = (10,10))
fig.suptitle("First 16 Images", fontsize = 16)

for i, ax in enumerate(axes.flat):
    img = df.iloc[i, 1:].values.reshape(28, 28)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"Label: {df.iloc[i, 0]}")
    

In [0]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [0]:
X_train = X_train/255
X_test = X_test/255

In [0]:
class CustomDataset(Dataset):
    
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype = torch.float32)
        self.labels = torch.tensor(labels, dtype = torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [0]:
train_dataset = CustomDataset(X_train, y_train)

In [0]:
test_dataset =CustomDataset(X_test, y_test)

In [0]:
train_loader = DataLoader(train_dataset, batch_size = 128, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 128, shuffle = True)

In [0]:
class MyNN(nn.Module):
    
    def __init__(self, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p = 0.5),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p = 0.3),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.network(x)

In [0]:
epochs = 100
learning_rate = 0.1

In [0]:
model = MyNN(X_train.shape[1])

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr = learning_rate, weight_decay = 1e-2)

In [0]:
len(train_loader)

In [0]:
for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_epoch_loss = total_epoch_loss + loss.item()

    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')
        

In [0]:
model.eval()

In [0]:
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        output = model(batch_features)
        predicted = torch.argmax(output, dim = 1)
        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()
    print(correct/total)

In [0]:
# evaluation on training data
total = 0
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    # move data to gpu
    outputs = model(batch_features)
    _, predicted = torch.max(outputs, 1)
    total = total + batch_labels.shape[0]
    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)